<a href="https://colab.research.google.com/github/Mohanapriya2210/Finalyearproject/blob/main/dr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# STEP 0: Imports & Drive
# =========================

from google.colab import drive
drive.mount('/content/drive')

import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# =========================
# STEP 1: Paths & CSVs
# =========================

BASE_PATH = "/content/drive/MyDrive/APTOS_FULL"

TRAIN_CSV = os.path.join(BASE_PATH, "train_1.csv")
VAL_CSV   = os.path.join(BASE_PATH, "valid.csv")
TEST_CSV  = os.path.join(BASE_PATH, "test.csv")

TRAIN_IMG_DIR = os.path.join(BASE_PATH, "train_images", "train_images")
VAL_IMG_DIR   = os.path.join(BASE_PATH, "val_images", "val_images")
TEST_IMG_DIR  = os.path.join(BASE_PATH, "test_images", "test_images")

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)


Train: (2930, 2)
Val: (366, 2)
Test: (366, 2)


In [ ]:
# =========================
# STEP 2: EfficientNet Feature Extractor
# =========================

base_model = EfficientNetV2S(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

# Freeze ALL layers (feature extraction only)
for layer in base_model.layers:
    layer.trainable = False

x = GlobalAveragePooling2D()(base_model.output)

feature_extractor = Model(
    inputs=base_model.input,
    outputs=x
)

print("Feature dimension:", feature_extractor.output_shape)


82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Feature dimension: (None, 1280)


In [ ]:
# =========================
# STEP 3: Streaming feature extraction
# =========================

def extract_and_save_features(
    model,
    df,
    image_dir,
    save_prefix,
    batch_size=8
):
    features = []
    labels = []

    batch_images = []
    batch_labels = []

    for i, row in enumerate(df.itertuples(), start=1):
        img_path = os.path.join(image_dir, row.id_code + ".png")

        if not os.path.exists(img_path):
            continue

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))
        img = img / 255.0

        batch_images.append(img)
        batch_labels.append(row.diagnosis)

        if len(batch_images) == batch_size:
            batch_images = np.array(batch_images)
            batch_features = model.predict(batch_images, verbose=0)

            features.append(batch_features)
            labels.extend(batch_labels)

            batch_images = []
            batch_labels = []

        if i % 200 == 0:
            print(f"Processed {i} images")

    # leftover images
    if len(batch_images) > 0:
        batch_images = np.array(batch_images)
        batch_features = model.predict(batch_images, verbose=0)
        features.append(batch_features)
        labels.extend(batch_labels)

    features = np.vstack(features)
    labels = np.array(labels)

    np.save(save_prefix + "_X.npy", features)
    np.save(save_prefix + "_y.npy", labels)

    print("Saved:", save_prefix)
    print("Shape:", features.shape)


In [ ]:
FEATURE_DIR = "/content/drive/MyDrive/APTOS_FEATURES"
os.makedirs(FEATURE_DIR, exist_ok=True)

extract_and_save_features(
    feature_extractor,
    train_df,
    TRAIN_IMG_DIR,
    os.path.join(FEATURE_DIR, "train"),
    batch_size=8
)

extract_and_save_features(
    feature_extractor,
    val_df,
    VAL_IMG_DIR,
    os.path.join(FEATURE_DIR, "val"),
    batch_size=8
)

extract_and_save_features(
    feature_extractor,
    test_df,
    TEST_IMG_DIR,
    os.path.join(FEATURE_DIR, "test"),
    batch_size=8
)


Processed 200 images
Processed 400 images
Processed 600 images
Processed 800 images
Processed 1000 images
Processed 1200 images
Processed 1400 images
Processed 1600 images
Processed 1800 images
Processed 2000 images
Processed 2200 images
Processed 2400 images
Processed 2600 images
Processed 2800 images
Saved: /content/drive/MyDrive/APTOS_FEATURES/train
Shape: (2930, 1280)
Processed 200 images
Saved: /content/drive/MyDrive/APTOS_FEATURES/val
Shape: (366, 1280)
Processed 200 images
Saved: /content/drive/MyDrive/APTOS_FEATURES/test
Shape: (366, 1280)


In [ ]:
# =========================
# STEP 4A: Load cached image features
# =========================

import numpy as np

FEATURE_DIR = "/content/drive/MyDrive/APTOS_FEATURES"

X_img_train = np.load(FEATURE_DIR + "/train_X.npy")
y_train = np.load(FEATURE_DIR + "/train_y.npy")

X_img_val = np.load(FEATURE_DIR + "/val_X.npy")
y_val = np.load(FEATURE_DIR + "/val_y.npy")

X_img_test = np.load(FEATURE_DIR + "/test_X.npy")
y_test = np.load(FEATURE_DIR + "/test_y.npy")

print("Train image features:", X_img_train.shape)
print("Val image features:", X_img_val.shape)
print("Test image features:", X_img_test.shape)


Train image features: (2930, 1280)
Val image features: (366, 1280)
Test image features: (366, 1280)


In [ ]:
import pandas as pd

pima_path = "/content/drive/MyDrive/diabetes.csv"  # adjust if needed
pima = pd.read_csv(pima_path)

print("Original PIMA shape:", pima.shape)
pima.head()


Original PIMA shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [ ]:
cols_with_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
pima[cols_with_zeros] = pima[cols_with_zeros].replace(0, np.nan)
pima = pima.dropna().reset_index(drop=True)

print("After cleaning:", pima.shape)


After cleaning: (392, 9)


In [ ]:
X_tab = pima.drop(columns=['Outcome'])


In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler_tab = MinMaxScaler()
X_tab_scaled = scaler_tab.fit_transform(X_tab)

print("Tabular feature shape:", X_tab_scaled.shape)


Tabular feature shape: (392, 8)


In [ ]:
glucose = X_tab['Glucose'].values

low_idx    = np.where(glucose < 110)[0]
mid_idx    = np.where((glucose >= 110) & (glucose < 140))[0]
high_idx   = np.where(glucose >= 140)[0]

print("Low glucose samples:", len(low_idx))
print("Mid glucose samples:", len(mid_idx))
print("High glucose samples:", len(high_idx))


Low glucose samples: 161
Mid glucose samples: 123
High glucose samples: 108


In [ ]:
import numpy as np

def sample_tabular_by_severity(severity, X_tab_scaled):
    if severity == 0:
        choices = low_idx
    elif severity == 1:
        choices = np.concatenate([low_idx, mid_idx])
    elif severity == 2:
        choices = mid_idx
    elif severity == 3:
        choices = np.concatenate([mid_idx, high_idx])
    else:  # severity == 4
        choices = high_idx

    idx = np.random.choice(choices)
    return X_tab_scaled[idx]


In [ ]:
# =========================
# STEP 4D: Align tabular data with image severity
# =========================

X_tab_aligned = np.array([
    sample_tabular_by_severity(sev, X_tab_scaled)
    for sev in y_train
])

print("Aligned tabular shape:", X_tab_aligned.shape)


Aligned tabular shape: (2930, 8)


In [ ]:
# =========================
# STEP 5A: Tabular embedding network
# =========================

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

tab_input = Input(shape=(X_tab_aligned.shape[1],))  # 8 features

x = Dense(64, activation='relu')(tab_input)
x = Dense(256, activation='relu')(x)
tab_embedding = Dense(1280, activation='relu')(x)

tab_model = Model(inputs=tab_input, outputs=tab_embedding)
tab_model.summary()


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 256)            │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1280)           │       328,960 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 346,176 (1.32 MB)

 Trainable params: 346,176 (1.32 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# =========================
# STEP 5B: Generate tabular embeddings
# =========================

X_tab_embed = tab_model.predict(
    X_tab_aligned,
    batch_size=32,
    verbose=1
)

print("Tabular embeddings shape:", X_tab_embed.shape)


92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Tabular embeddings shape: (2930, 1280)


In [ ]:
# =========================
# STEP 6A: Regularized gated fusion
# =========================

from tensorflow.keras.layers import (
    Input, Dense, Concatenate, Multiply, Add, Lambda, Dropout
)
from tensorflow.keras.models import Model
from tensorflow.keras import regularizers

# Inputs
img_input = Input(shape=(1280,), name="image_features")
tab_input = Input(shape=(1280,), name="tabular_features")

# Concatenate for gate learning
concat = Concatenate()([img_input, tab_input])

# Learn gate (with regularization)
gate = Dense(
    1280,
    activation='sigmoid',
    kernel_regularizer=regularizers.l2(1e-4)
)(concat)

# Inverse gate: (1 - gate)
inv_gate = Lambda(lambda x: 1.0 - x)(gate)

# Weighted modalities
weighted_img = Multiply()([gate, img_input])
weighted_tab = Multiply()([inv_gate, tab_input])

# Fuse
fusion = Add()([weighted_img, weighted_tab])

# Optional dropout for stability
fusion = Dropout(0.3)(fusion)

# Fusion model
fusion_model = Model(
    inputs=[img_input, tab_input],
    outputs=fusion
)

fusion_model.summary()


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_features      │ (None, 1280)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tabular_features    │ (None, 1280)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 2560)      │          0 │ image_features[0… │
│ (Concatenate)       │                   │            │ tabular_features… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 1280)      │  3,278,080 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 1280)      │          0 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_2          │ (None, 1280)      │          0 │ dense_7[0][0],    │
│ (Multiply)          │                   │            │ image_features[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_3          │ (None, 1280)      │          0 │ lambda_1[0][0],   │
│ (Multiply)          │                   │            │ tabular_features… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 1280)      │          0 │ multiply_2[0][0], │
│                     │                   │            │ multiply_3[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 1280)      │          0 │ add_1[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,278,080 (12.50 MB)

 Trainable params: 3,278,080 (12.50 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# =========================
# STEP 6B: Generate fused features
# =========================

X_fused = fusion_model.predict(
    [X_img_train, X_tab_embed],
    batch_size=32,
    verbose=1
)

print("Fused feature shape:", X_fused.shape)


92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fused feature shape: (2930, 1280)


In [ ]:
# =========================
# STEP 6C: Scale fused features
# =========================

from sklearn.preprocessing import StandardScaler

scaler_fused = StandardScaler()
X_fused_scaled = scaler_fused.fit_transform(X_fused)

print("Scaled fused features:", X_fused_scaled.shape)


Scaled fused features: (2930, 1280)


In [ ]:
# =========================
# STEP 7A: Train-validation split
# =========================

from sklearn.model_selection import train_test_split

X_tr, X_val2, y_tr, y_val2 = train_test_split(
    X_fused_scaled,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

print(X_tr.shape, X_val2.shape)


(2344, 1280) (586, 1280)


In [ ]:
# =========================
# STEP 7B: XGBoost training
# =========================

from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42
)

xgb_model.fit(X_tr, y_tr)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None, num_class=5, ...)

In [ ]:
# =========================
# STEP 7C: Validation evaluation
# =========================

from sklearn.metrics import classification_report, confusion_matrix

y_val_pred = xgb_model.predict(X_val2)

print("Validation Classification Report:\n")
print(classification_report(y_val2, y_val_pred))

print("Confusion Matrix:\n")
print(confusion_matrix(y_val2, y_val_pred))


Validation Classification Report:

              precision    recall  f1-score   support

           0       0.93      0.96      0.94       287
           1       0.50      0.33      0.40        60
           2       0.67      0.88      0.76       161
           3       0.75      0.10      0.17        31
           4       0.77      0.57      0.66        47

    accuracy                           0.80       586
   macro avg       0.72      0.57      0.59       586
weighted avg       0.79      0.80      0.77       586

Confusion Matrix:

[[275   6   6   0   0]
 [ 12  20  28   0   0]
 [  8  11 141   0   1]
 [  1   2  18   3   7]
 [  0   1  18   1  27]]


In [ ]:
# =========================
# STEP 7D: Prepare TEST fused features
# =========================

# Align tabular features for test labels
X_tab_test_aligned = np.array([
    sample_tabular_by_severity(sev, X_tab_scaled)
    for sev in y_test
])

# Embed tabular test features
X_tab_test_embed = tab_model.predict(X_tab_test_aligned, verbose=0)

# Fuse
X_fused_test = fusion_model.predict(
    [X_img_test, X_tab_test_embed],
    verbose=0
)

# Scale using TRAIN scaler
X_fused_test_scaled = scaler_fused.transform(X_fused_test)


In [ ]:
# =========================
# STEP 7E: TEST evaluation
# =========================

y_test_pred = xgb_model.predict(X_fused_test_scaled)

print("TEST Classification Report:\n")
print(classification_report(y_test, y_test_pred))

print("TEST Confusion Matrix:\n")
print(confusion_matrix(y_test, y_test_pred))


TEST Classification Report:

              precision    recall  f1-score   support

           0       0.95      0.96      0.96       199
           1       0.46      0.40      0.43        30
           2       0.64      0.86      0.73        87
           3       0.40      0.12      0.18        17
           4       0.88      0.45      0.60        33

    accuracy                           0.81       366
   macro avg       0.67      0.56      0.58       366
weighted avg       0.81      0.81      0.79       366

TEST Confusion Matrix:

[[191   2   6   0   0]
 [  5  12  13   0   0]
 [  4   8  75   0   0]
 [  0   3  10   2   2]
 [  0   1  14   3  15]]


In [ ]:
import os, joblib

MODEL_DIR = "/content/drive/MyDrive/DR_MODEL"
os.makedirs(MODEL_DIR, exist_ok=True)

tab_model.save("/content/drive/MyDrive/DR_MODEL/tab_embedding_model.keras")
fusion_model.save("/content/drive/MyDrive/DR_MODEL/fusion_model.keras")

joblib.dump(xgb_model, "/content/drive/MyDrive/DR_MODEL/xgb_model.pkl")
joblib.dump(scaler_tab, "/content/drive/MyDrive/DR_MODEL/tab_scaler.pkl")
joblib.dump(scaler_fused, "/content/drive/MyDrive/DR_MODEL/fusion_scaler.pkl")

print("✅ All models and scalers saved successfully")


✅ All models and scalers saved successfully
